# Beispiel-Notebook: Nutzung des Time-Expanded Modells für Fracht-Routing

---

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from freight_routing.data_loader import NetworkDataLoader
from freight_routing.data_models import Shipment, ObjectiveWeights
from freight_routing.model import TimeExpandedFreightRoutingModel, TimeExpandedNetwork
from freight_routing.visualization import create_network_map

## 1. Laden des Transportnetzwerks

In [2]:
test_network_path = PROJECT_ROOT / "dataset" / "test_network.json"
network_data = NetworkDataLoader.from_json(test_network_path)

network_data.summary()

Summary NetworkData:
hubs=3
arcs=6
modes=2


## 2. Definition der Sendungen (Shipments)

In [3]:
shipment = Shipment(
    id="sendung_ber_muc_1",
    start_hub="BER",
    end_hub="MUC",
    start_time=0,
    deadline=2880,  # 48 Stunden Frist
    max_price=5000.0,  # Budgetgrenze
    max_emissions=None,  # Keine Emissionsgrenze
    weight=5.0,  # 5 Tonnen Gewicht
)

shipments = [shipment]
print(f"Definierte Sendung: {shipment}")

Definierte Sendung: Shipment(id='sendung_ber_muc_1', start_hub='BER', end_hub='MUC', start_time=0, deadline=2880, max_price=5000.0, max_emissions=None, weight=5.0)


## 3. Modell instanziieren, bauen und lösen

In [4]:
network = TimeExpandedNetwork.build(network_data, planning_days=2, shipments=shipments)
model = TimeExpandedFreightRoutingModel()
network.summary()

result = model.solve(network)

print("\n--- Optimierungsergebnisse ---")
print(f"Solver Status: {result.status}")
print(f"Optimalität erreicht: {result.is_optimal}")

Summary TimeExpandedNetwork:
planning_days=2
planning_horizon_min=2880
nodes=48
arcs=64
  - transport_arcs=10
  - transfer_arcs=12
  - waiting_arcs=42

--- Optimierungsergebnisse ---
Solver Status: Optimal
Optimalität erreicht: True


## 4. Analyse der Ergebnisse und Routenverlauf

In [5]:
if result.is_optimal:
    print(
        f"Gesamtkosten: {result.total_cost:.2f} EUR (Fix: {result.total_fixed_cost:.2f} EUR, Variabel: {result.total_variable_cost:.2f} EUR)"
    )
    print(
        f"Gesamtemissionen: {result.total_emissions:.2f} kg CO2 (Fix: {result.total_fixed_emissions:.2f} kg, Variabel: {result.total_variable_emissions:.2f} kg)"
    )
    print(
        f"Gesamtzeit: {result.total_time} Minuten ({result.total_time / 60:.2f} Stunden)"
    )

    print("\nGewählter Routenverlauf:")
    route = result.shipment_routes[shipment.id]
    for i, arc in enumerate(route, 1):
        print(f"  Schritt {i}:")
        print(
            f"    Von: {arc.from_node.hub_id} ({arc.from_node.mode}) zur Zeit {arc.from_node.time_min} min"
        )
        print(
            f"    Nach: {arc.to_node.hub_id} ({arc.to_node.mode}) zur Zeit {arc.to_node.time_min} min"
        )
        print(
            f"    Typ: {arc.arc_type.value} | Distanz: {arc.distance} km | Modus: {arc.mode}"
        )
        print(
            f"    Kosten: {arc.cost:.2f} EUR | Emissionen: {arc.emissions:.2f} kg CO2"
        )
else:
    print(
        "Keine zulässige Route gefunden, die alle Nebenbedingungen (z.B. Deadlines) einhält."
    )

Gesamtkosten: 960.00 EUR (Fix: 200.00 EUR, Variabel: 760.00 EUR)
Gesamtemissionen: 382.00 kg CO2 (Fix: 20.00 kg, Variabel: 362.00 kg)
Gesamtzeit: 1260.0 Minuten (21.00 Stunden)

Gewählter Routenverlauf:
  Schritt 1:
    Von: BER (road) zur Zeit 0 min
    Nach: BER (road) zur Zeit 360 min
    Typ: waiting | Distanz: 0.0 km | Modus: road
    Kosten: 12.00 EUR | Emissionen: 0.30 kg CO2
  Schritt 2:
    Von: BER (road) zur Zeit 360 min
    Nach: HAM (road) zur Zeit 600 min
    Typ: transport | Distanz: 290.0 km | Modus: road
    Kosten: 43.50 EUR | Emissionen: 23.20 kg CO2
  Schritt 3:
    Von: HAM (road) zur Zeit 600 min
    Nach: HAM (road) zur Zeit 720 min
    Typ: waiting | Distanz: 0.0 km | Modus: road
    Kosten: 5.00 EUR | Emissionen: 0.10 kg CO2
  Schritt 4:
    Von: HAM (road) zur Zeit 720 min
    Nach: MUC (road) zur Zeit 1260 min
    Typ: transport | Distanz: 610.0 km | Modus: road
    Kosten: 91.50 EUR | Emissionen: 48.80 kg CO2


## 5. Kosten vs. Emissionen: Gewichtung anpassen

1. **Szenario A: Reine Kostenoptimierung** (Kostengewicht = 1.0, Emissionsgewicht = 0.0)
2. **Szenario B: Reine Emissionsoptimierung** (Kostengewicht = 0.0, Emissionsgewicht = 1.0)

In [6]:
# Szenario A: Kosten im Fokus
weights_cost = ObjectiveWeights(cost=1.0, emissions=0.0, time=0.0)
net_cost = TimeExpandedNetwork.build(network_data, planning_days=2, shipments=shipments)
model_cost = TimeExpandedFreightRoutingModel(objective_weights=weights_cost)
result_cost = model_cost.solve(net_cost)

# Szenario B: Emissionen im Fokus
weights_eco = ObjectiveWeights(cost=0.0, emissions=1.0, time=0.0)
net_eco = TimeExpandedNetwork.build(network_data, planning_days=2, shipments=shipments)
model_eco = TimeExpandedFreightRoutingModel(objective_weights=weights_eco)
result_eco = model_eco.solve(net_eco)

print("=== Szenario A: Kostenoptimiert ===")
if result_cost.is_optimal:
    print(f"Kosten: {result_cost.total_cost:.2f} EUR")
    print(f"Emissionen: {result_cost.total_emissions:.2f} kg CO2")
    route_cost = result_cost.shipment_routes[shipment.id]

=== Szenario A: Kostenoptimiert ===
Kosten: 960.00 EUR
Emissionen: 382.00 kg CO2


## 6. Routen-Visualisierung auf einer interaktiven Karte

Mit der Funktion `create_network_map` können wir die static Verbindungen des multimodalen Netzwerks sowie die berechnete Route auf einer interaktiven Folium-Karte darstellen.

In [7]:
if result_eco.is_optimal:
    m = create_network_map(
        network_data, route=result_eco.shipment_routes[shipment.id], show_network=True
    )
    display(m)
else:
    print("Keine Route zur Visualisierung verfügbar.")